# Exploratory Data Analysis (EDA) for IoT Sensor Anomaly Detection**Project:** AI-Based IoT Sensor Anomaly Detection**Sensor:** ESP32 + DHT11 (GPIO 4)**Datasets:** normal_data.csv, pilot_data.csv, test.csv, test_labels.csv---

## 1. Setup and Data LoadingWe first import necessary libraries and load all four datasets. Note that `normal_data.csv` **does not have a header row** in the original file, while the other three datasets do.

In [1]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Path# Set style for plotsplt.style.use('seaborn-v0_8-whitegrid')sns.set_palette('husl')# Create plots directoryplots_dir = Path('eda/plots')plots_dir.mkdir(parents=True, exist_ok=True)# Load datasets# normal_data.csv has NO header - first row is datanormal = pd.read_csv('data/raw/normal_data.csv', header=None, names=['timestamp', 'temperature', 'humidity'])pilot = pd.read_csv('data/raw/pilot_data.csv')test = pd.read_csv('data/raw/test.csv')test_labels = pd.read_csv('data/raw/test_labels.csv')# Convert timestamps to datetimenormal['timestamp'] = pd.to_datetime(normal['timestamp'])pilot['timestamp'] = pd.to_datetime(pilot['timestamp'])test['timestamp'] = pd.to_datetime(test['timestamp'])# Basic info functiondef dataset_info(df, name):    print(f'=== {name} ===')    print(f'Shape: {df.shape}')    print(f'Columns: {list(df.columns)}')    print(f'Dtypes:\n{df.dtypes}')    print(f'Missing:\n{df.isnull().sum()}')    print(f'Duplicates (all cols): {df.duplicated().sum()}')    if 'timestamp' in df.columns:        print(f'Duplicate timestamps: {df.duplicated(subset=["timestamp"]).sum()}')    print(f'First 5:\n{df.head()}')    print(f'Last 5:\n{df.tail()}')    if 'timestamp' in df.columns:        print(f'Timestamp range: {df["timestamp"].min()} to {df["timestamp"].max()}')    print()dataset_info(normal, 'NORMAL_DATA.CSV')dataset_info(pilot, 'PILOT_DATA.CSV')dataset_info(test, 'TEST.CSV')dataset_info(test_labels, 'TEST_LABELS.CSV')

=== NORMAL_DATA.CSV ===Shape: (49728, 3)Columns: ['timestamp', 'temperature', 'humidity']Dtypes:timestamp          objecttemperature      float64humidity         float64dtype: objectMissing:timestamp      0temperature    0humidity       0dtype: int64Duplicates (all cols): 12776Duplicate timestamps: 13065First 5:             timestamp  temperature  humidity0  2026-10-29T16:39:00         25.8      60.71  2026-10-29T16:39:02         25.8      60.12  2026-10-29T16:39:04         25.8      59.43  2026-10-29T16:39:06         25.7      58.64  2026-10-29T16:39:08         25.7      57.7Last 5:                 timestamp  temperature  humidity49723  2026-11-01T17:52:24         26.3      58.249724  2026-11-01T17:52:26         26.3      58.249725  2026-11-01T17:52:28         26.3      58.249726  2026-11-01T17:52:30         26.3      58.249727  2026-11-01T17:52:32         26.3      58.2Timestamp range: 2026-10-29T16:39:00 to 2026-11-01T17:52:32=== PILOT_DATA.CSV ===Shape: (3087, 3)Columns: ['timestam

## 2. Dataset Inventory Summary| Dataset | Rows | Columns | Header | Duplicates (all) | Duplicate Timestamps | Date Range ||---------|------|---------|--------|------------------|---------------------|------------|| normal_data.csv | 49,728 | 3 | ❌ No | 12,776 | 13,065 | 2026-10-29 to 2026-11-01 (~3 days) || pilot_data.csv | 3,087 | 3 | ✅ Yes | 0 | 0 | 2026-10-27 to 2026-10-28 (~3 hours) || test.csv | 56 | 3 | ✅ Yes | 0 | 0 | 2026-10-27 22:02 to 22:04 (~2 min) || test_labels.csv | 1 | 3 | ✅ Yes | 0 | N/A | Single event label |### Key Observations:1. **normal_data.csv lacks a header row** - the first data row was being interpreted as column names. We manually assigned column names.2. **Massive duplication in normal_data.csv**: 13,065 duplicate timestamps (26.3% of rows), 12,776 exact duplicate rows.3. **pilot_data.csv and test.csv are clean** - no duplicates, proper headers.4. **test_labels.csv contains an EVENT-LEVEL label**, not row-level. It marks a time window (22:03:29 to 22:03:47) as "anomaly".

## 3. Statistical Analysis of normal_data.csv (Candidate Training Set)### What we're doing and why:- **Descriptive statistics** (mean, median, std, quartiles) tell us the central tendency and spread of normal sensor readings.- **IQR (Interquartile Range)** = Q3 - Q1 helps identify statistical outliers without assuming Gaussian distribution.- For anomaly detection, we need to know what "normal" looks like to detect deviations.### Python functions used:- `df.describe()` - computes count, mean, std, min, quartiles, max- `df.quantile([0.25, 0.5, 0.75])` - explicit quartile calculation- `df.var()` - variance

In [2]:
print('=== NORMAL_DATA.CSV - Temperature Statistics ===')print(normal['temperature'].describe())print(f'Variance: {normal["temperature"].var():.6f}')q1_temp = normal['temperature'].quantile(0.25)q3_temp = normal['temperature'].quantile(0.75)iqr_temp = q3_temp - q1_tempprint(f'IQR: {iqr_temp:.6f}')print(f'Lower fence (Q1 - 1.5*IQR): {q1_temp - 1.5*iqr_temp:.6f}')print(f'Upper fence (Q3 + 1.5*IQR): {q3_temp + 1.5*iqr_temp:.6f}')print()print('=== NORMAL_DATA.CSV - Humidity Statistics ===')print(normal['humidity'].describe())print(f'Variance: {normal["humidity"].var():.6f}')q1_hum = normal['humidity'].quantile(0.25)q3_hum = normal['humidity'].quantile(0.75)iqr_hum = q3_hum - q1_humprint(f'IQR: {iqr_hum:.6f}')print(f'Lower fence (Q1 - 1.5*IQR): {q1_hum - 1.5*iqr_hum:.6f}')print(f'Upper fence (Q3 + 1.5*IQR): {q3_hum + 1.5*iqr_hum:.6f}')

=== NORMAL_DATA.CSV - Temperature Statistics ===count    49728.000000mean        25.808518std          0.736721min         22.40000025%         25.80000050%         26.00000075%         26.200000max         27.000000Name: temperature, dtype: float64Variance: 0.542759IQR: 0.400000Lower fence (Q1 - 1.5*IQR): 25.200000Upper fence (Q3 + 1.5*IQR): 26.800000=== NORMAL_DATA.CSV - Humidity Statistics ===count    49728.000000mean        62.072977std          4.258885min         46.80000025%         61.00000050%         63.50000075%         64.400000max         97.200000Name: humidity, dtype: float64Variance: 18.138101IQR: 3.400000Lower fence (Q1 - 1.5*IQR): 55.900000Upper fence (Q3 + 1.5*IQR): 69.500000

### Interpretation:**Temperature:**- Mean: 25.81°C, Median: 26.0°C (slight left skew)- Range: 22.4°C to 27.0°C (4.6°C span)- IQR: 0.4°C (very tight - middle 50% of readings span only 0.4°C)- IQR fences: 25.2°C to 26.8°C- **Values below 25.2°C or above 26.8°C are statistical outliers** by IQR criterion- Minimum 22.4°C is well below lower fence - potential outliers or sensor issues**Humidity:**- Mean: 62.07%, Median: 63.5% (left skew)- Range: 46.8% to 97.2% (50.4% span - much wider than temperature)- IQR: 3.4% (middle 50% spans 3.4%)- IQR fences: 55.9% to 69.5%- **Maximum 97.2% is far above upper fence (69.5%)** - clear outliers- Minimum 46.8% is below lower fence (55.9%) - potential outliers**Key insight for ML:** Humidity has much higher variance (18.14 vs 0.54) and wider range. The model must handle this multivariate difference. The IQR outliers in humidity (especially the 97.2% max) warrant investigation - are they real humidity spikes or sensor artifacts?

## 4. Timestamp and Sampling Interval Analysis (normal_data.csv)### Why this matters:- The ESP32 is configured for 2000ms (2 second) sampling intervals- Deviations indicate: logging gaps, duplicate writes, system pauses, or clock issues- For time-series ML, regular sampling is often assumed; irregular sampling requires special handling### Python functions:- `df['timestamp'].diff().dt.total_seconds()` - computes time difference between consecutive rows- `value_counts()` - frequency of each interval value

In [3]:
normal['interval'] = normal['timestamp'].diff().dt.total_seconds()print('=== NORMAL_DATA.CSV - Sampling Interval Analysis ===')print(f'Total intervals: {len(normal["interval"].dropna())}')print(f'Mean: {normal["interval"].mean():.2f} seconds')print(f'Std: {normal["interval"].std():.2f} seconds')print(f'Min: {normal["interval"].min():.2f} seconds')print(f'25%: {normal["interval"].quantile(0.25):.2f} seconds')print(f'50%: {normal["interval"].quantile(0.50):.2f} seconds')print(f'75%: {normal["interval"].quantile(0.75):.2f} seconds')print(f'Max: {normal["interval"].max():.2f} seconds ({normal["interval"].max()/3600:.1f} hours)')print()print('Interval value counts:')print(normal['interval'].value_counts())

=== NORMAL_DATA.CSV - Sampling Interval Analysis ===Total intervals: 49727Mean: 5.30 secondsStd: 643.91 secondsMin: 0.00 seconds25%: 0.00 seconds50%: 2.00 seconds75%: 2.00 secondsMax: 142210.00 seconds (39.5 hours)Interval value counts:2.0         357270.0         130653.0           9141.0            116965.0          112689.0         11119.0          13285.0          1725.0           16550.0          13269.0          11842.0          110751.0         1142210.0        1Name: count, dtype: int64

### Interpretation:- **35,727 intervals (71.8%) are exactly 2.0 seconds** - matches expected ESP32 sampling rate ✓- **13,065 intervals (26.3%) are 0.0 seconds** - duplicate timestamps! This means multiple rows logged at the exact same second.- **914 intervals (1.8%) are 3.0 seconds** - slight delays, possibly garbage collection or serial buffer flush- **11 intervals of 1.0 second** - timing jitter- **7 large gaps**: 725s (12 min), 1,119s (18 min), 1,842s (30 min), 3,269s (54 min), 3,285s (55 min), 6,550s (1.8 hrs), 6,965s (1.9 hrs), 10,751s (3 hrs), 12,689s (3.5 hrs), 142,210s (39.5 hrs)**The 39.5-hour gap is particularly concerning** - it likely represents a period when the logger was stopped or the ESP32 was powered off.**Implication for ML:** The duplicate timestamps (0-second intervals) mean we have multiple readings for the same moment. This could bias training if not handled. The large gaps mean the data is not continuous - there are significant missing periods.

## 5. Duplicate Analysis (normal_data.csv)### Three types of duplicates to distinguish:1. **Exact duplicate rows**: identical timestamp, temperature, AND humidity2. **Repeated timestamps with different readings**: same timestamp, different temperature/humidity3. **Identical readings at different timestamps**: same temperature/humidity, different timestamps (normal for stable environment)### Why this matters:- Exact duplicates add no information but increase training weight for those values- Repeated timestamps with different values suggest logging race conditions or buffer issues- We must NOT automatically delete - need to understand the cause first

In [4]:
# Exact duplicatesexact_dup_count = normal.duplicated().sum()ts_dup_count = normal.duplicated(subset=['timestamp']).sum()th_dup_count = normal.duplicated(subset=['temperature', 'humidity']).sum()print('=== DUPLICATE ANALYSIS ===')print(f'Exact duplicate rows (all 3 cols): {exact_dup_count}')print(f'Duplicate timestamps: {ts_dup_count}')print(f'Duplicate temp+humidity (different timestamps): {th_dup_count}')print()# Repeated timestamps with different valuests_dups = normal[normal.duplicated(subset=['timestamp'], keep=False)].sort_values('timestamp')print('--- Repeated timestamps with different values (first 20) ---')print(ts_dups.head(20))print()# How many rows per duplicate timestamp?ts_counts = ts_dups.groupby('timestamp').size().sort_values(ascending=False)print('--- Count of rows per duplicate timestamp (top 10) ---')print(ts_counts.head(10))

=== DUPLICATE ANALYSIS ===Exact duplicate rows (all 3 cols): 12761Duplicate timestamps: 13065Duplicate temp+humidity (different timestamps): 48574--- Repeated timestamps with different values (first 20) ---                timestamp  temperature  humidity  interval34188 2026-10-30 18:33:40         26.0      62.0    3285.034263 2026-10-30 18:33:40         26.1      62.2       0.034262 2026-10-30 18:33:40         26.0      62.2       0.034261 2026-10-30 18:33:40         26.1      62.2       0.034260 2026-10-30 18:33:40         26.1      62.2       0.034259 2026-10-30 18:33:40         26.1      62.2       0.034258 2026-10-30 18:33:40         26.1      62.2       0.034257 2026-10-30 18:33:40         26.1      62.2       0.034256 2026-10-30 18:33:40         26.1      62.2       0.034255 2026-10-30 18:33:40         26.0      62.2       0.034254 2026-10-30 18:33:40         26.0      62.2       0.034253 2026-10-30 18:33:40         26.1      62.2       0.034252 2026-10-30 18:33:40         26.0  

### Interpretation:- **12,761 exact duplicate rows** - identical timestamp, temp, humidity. These are pure logging duplicates.- **13,065 duplicate timestamps** - but only 12,761 exact duplicates. The difference (304) are timestamps with **different sensor values**.- **Top duplicate timestamp has 80 rows at the exact same second** (2026-10-30 18:33:40)!- The next several have 53-54 rows each at the same timestamp.- This pattern (50-80 rows per timestamp) suggests a **logging bug** - possibly the Python logger wrote the same buffer multiple times, or the ESP32 sent burst data.**Critical for ML:** These duplicates will severely bias any model trained on this data. The model will see certain (timestamp, temp, humidity) combinations 50-80x more often than they actually occurred.

## 6. Analysis of pilot_data.csv (Contains Anomaly Events)### Why analyze pilot data:- Contains **real anomaly events** (not synthetic)- Helps us understand what anomalies look like in this sensor system- **Must NOT be used for normal-only training**- Useful for validation and threshold tuning

In [5]:
print('=== PILOT_DATA.CSV - Statistical Analysis ===')print('Temperature:')print(pilot['temperature'].describe())print()print('Humidity:')print(pilot['humidity'].describe())print()pilot['interval'] = pilot['timestamp'].diff().dt.total_seconds()print('Sampling interval stats (seconds):')print(pilot['interval'].describe())print()print('Interval value counts:')print(pilot['interval'].value_counts())

=== PILOT_DATA.CSV - Statistical Analysis ===Temperature:count    3087.000000mean       27.291480std         1.470235min        26.00000025%        26.60000050%        26.80000075%        26.900000max        32.900000Name: temperature, dtype: float64Humidity:count    3087.000000mean       57.540460std         5.231717min        44.40000025%        57.60000050%        58.80000075%        60.000000max        95.800000Name: humidity, dtype: float64Sampling interval stats (seconds):count    3086.000000mean        3.568697std        71.288154min         2.00000025%         2.00000050%         2.00000075%         2.000000Max      3857.000000Interval value counts:interval2.0       30063.0         783857.0       1910.0        1Name: count, dtype: int64

### Pilot Data - Anomaly DetectionLet's identify the anomaly periods using statistical methods and sudden change analysis.

In [6]:
# Statistical outlier detection (3-sigma)temp_mean = pilot['temperature'].mean()temp_std = pilot['temperature'].std()hum_mean = pilot['humidity'].mean()hum_std = pilot['humidity'].std()print(f'Temperature mean: {temp_mean:.2f}, std: {temp_std:.2f}')high_temp = pilot[pilot['temperature'] > temp_mean + 3*temp_std]print('High temperature outliers (>3 std):')print(high_temp[['timestamp', 'temperature', 'humidity']])print()print(f'Humidity mean: {hum_mean:.2f}, std: {hum_std:.2f}')high_hum = pilot[pilot['humidity'] > hum_mean + 3*hum_std]print('High humidity outliers (>3 std):')print(high_hum[['timestamp', 'temperature', 'humidity']])print()# Sudden change analysispilot['temp_diff'] = pilot['temperature'].diff().abs()pilot['hum_diff'] = pilot['humidity'].diff().abs()print('Largest temperature changes:')print(pilot.nlargest(5, 'temp_diff')[['timestamp', 'temperature', 'temp_diff']])print()print('Largest humidity changes:')print(pilot.nlargest(5, 'hum_diff')[['timestamp', 'humidity', 'hum_diff']])

Temperature mean: 27.29, std: 1.47High temperature outliers (>3 std):               timestamp  temperature  humidity416  2026-10-27 23:20:32         31.8      45.2417  2026-10-27 23:20:34         31.8      45.22156 2026-10-28 00:34:24         32.4      95.02157 2026-10-28 00:34:26         32.9      95.02158 2026-10-28 00:34:28         32.7      95.02159 2026-10-28 00:34:30         32.5      95.0Humidity mean: 57.54, std: 5.23High humidity outliers (>3 std):               timestamp  temperature  humidity93   2026-10-27 23:09:38         26.6      85.194   2026-10-27 23:09:40         27.1      95.295   2026-10-27 23:09:42         27.3      93.196   2026-10-27 23:09:44         27.4      90.597   2026-10-27 23:09:46         27.4      88.998   2026-10-27 23:09:48         27.4      87.999   2026-10-27 23:09:50         27.4      82.62154 2026-10-28 00:34:20         28.6      95.02155 2026-10-28 00:34:22         31.3      95.02156 2026-10-28 00:34:24         32.4      95.02157 2026-10-28 00:34:

### Pilot Data - Anomaly Events Identified**Two clear anomaly events:****Event 1: 2026-10-27 23:09:38 - 23:10:30** (approx 52 seconds)- Temperature: rises from ~26.1°C to 27.4°C (+1.3°C)- Humidity: **massive spike from ~58% to 95.2%** (+37%!)- Largest humidity jump: 25.5% in 2 seconds (row 93)- Returns to normal gradually over ~50 seconds**Event 2: 2026-10-28 00:34:20 - 00:34:56** (approx 36 seconds)- Temperature: jumps from ~26.8°C to 32.9°C (+6.1°C!)- Humidity: jumps from ~57% to 95.8% (+38%!)- Largest humidity jump: 37.3% in 2 seconds (row 2154)- Largest temperature jump: 4.4°C in 2 seconds (row 2160)- Both temp and humidity spike simultaneously**Also noted:** A 3,857-second gap (~64 minutes) at some point in the data.**Interpretation:** These look like real environmental anomalies (possibly steam, cooking, shower, or sensor exposure to moisture/heat). The simultaneous temperature AND humidity spikes suggest a real physical event, not just sensor noise.

## 7. Analysis of test.csv and test_labels.csv### Critical: Label alignment- test_labels.csv has **1 row** with an event-level label: `anomaly` from 22:03:29 to 22:03:47- test.csv has **56 rows** total- **10 rows fall within the label window** (indices 37-46)- **46 rows are normal** (outside the window)- The label is **event-level**, NOT row-level### This means:- We cannot simply use test_labels.csv as y_true for row-wise classification- We must expand the event label to row-level for evaluation- The test set is small (56 rows) - only suitable for final evaluation, not threshold tuning

In [7]:
print('=== TEST.CSV - Statistical Analysis ===')print('Temperature:')print(test['temperature'].describe())print()print('Humidity:')print(test['humidity'].describe())print()test['interval'] = test['timestamp'].diff().dt.total_seconds()print('Sampling interval stats (seconds):')print(test['interval'].describe())print()print('=== TEST_LABELS.CSV Analysis ===')print(test_labels)label_start = pd.to_datetime(test_labels['start'].iloc[0])label_end = pd.to_datetime(test_labels['end'].iloc[0])print(f'Label window: {label_start} to {label_end}')# Which test rows fall in the label window?in_window = test[(test['timestamp'] >= label_start) & (test['timestamp'] <= label_end)]print(f'Test rows in label window: {len(in_window)}')print(in_window[['timestamp', 'temperature', 'humidity']])print()out_window = test[(test['timestamp'] < label_start) | (test['timestamp'] > label_end)]print(f'Test rows outside label window: {len(out_window)}')

=== TEST.CSV - Statistical Analysis ===Temperature:count    56.000000mean     27.114286std       2.358692min      26.20000025%      26.20000050%      26.20000075%      26.200000max      34.200000Name: temperature, dtype: float64Humidity:count    56.000000mean     62.128571std       6.579824min      59.30000025%      59.40000050%      59.50000075%      59.625000max      82.100000Sampling interval stats (seconds):count    55.000000mean      2.018182std       0.134840min       2.00000025%       2.00000050%       2.00000075%       2.000000Max       3.000000=== TEST_LABELS.CSV Analysis ===                 start                  end    label0  2026-10-27T22:03:29  2026-10-27T22:03:47  anomalyLabel window: 2026-10-27 22:03:29 to 2026-10-27 22:03:47Test rows in label window: 10             timestamp  temperature  humidity37 2026-10-27 22:03:29         32.5      75.038 2026-10-27 22:03:31         33.1      78.239 2026-10-27 22:03:33         33.8      80.540 2026-10-27 22:03:35         34.2     

### Test Data Anomaly Details:The injected anomaly (rows 37-46):- Temperature spikes from 26.2°C → 34.2°C (+8.0°C) then back to 26.3°C- Humidity spikes from 59.4% → 82.1% (+22.7%) then back to 60.2%- Duration: 18 seconds (10 readings at ~2s interval)- This is a **clear, sharp anomaly** - much more pronounced than pilot data events**Important:** The normal baseline in test.csv (26.2°C, 59.4%) matches the pilot_data.csv normal baseline (~26.2-26.8°C, ~59-60%), but differs from normal_data.csv (mean 25.8°C, 62.1%). This suggests different environmental conditions during different collection sessions.

## 8. VisualizationsNow we generate plots for each dataset. Each plot will have proper titles, axis labels, and legends.

In [8]:
# ============================================================# VISUALIZATIONS FOR normal_data.csv# ============================================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# Temperature histogramaxes[0, 0].hist(normal['temperature'], bins=50, alpha=0.7, edgecolor='black', color='skyblue')axes[0, 0].axvline(normal['temperature'].mean(), color='red', linestyle='--', label=f'Mean: {normal["temperature"].mean():.2f}')axes[0, 0].axvline(normal['temperature'].median(), color='green', linestyle='--', label=f'Median: {normal["temperature"].median():.2f}')axes[0, 0].set_title('Temperature Distribution (normal_data.csv)')axes[0, 0].set_xlabel('Temperature (°C)')axes[0, 0].set_ylabel('Frequency')axes[0, 0].legend()# Humidity histogramaxes[0, 1].hist(normal['humidity'], bins=50, alpha=0.7, edgecolor='black', color='lightcoral')axes[0, 1].axvline(normal['humidity'].mean(), color='red', linestyle='--', label=f'Mean: {normal["humidity"].mean():.2f}')axes[0, 1].axvline(normal['humidity'].median(), color='green', linestyle='--', label=f'Median: {normal["humidity"].median():.2f}')axes[0, 1].set_title('Humidity Distribution (normal_data.csv)')axes[0, 1].set_xlabel('Humidity (% RH)')axes[0, 1].set_ylabel('Frequency')axes[0, 1].legend()# Temperature boxplotaxes[1, 0].boxplot(normal['temperature'], vert=True, patch_artist=True,                   boxprops=dict(facecolor='skyblue'),                   medianprops=dict(color='red'))axes[1, 0].set_title('Temperature Boxplot (normal_data.csv)')axes[1, 0].set_ylabel('Temperature (°C)')axes[1, 0].set_xticklabels(['Temperature'])# Humidity boxplotaxes[1, 1].boxplot(normal['humidity'], vert=True, patch_artist=True,                   boxprops=dict(facecolor='lightcoral'),                   medianprops=dict(color='red'))axes[1, 1].set_title('Humidity Boxplot (normal_data.csv)')axes[1, 1].set_ylabel('Humidity (% RH)')axes[1, 1].set_xticklabels(['Humidity'])plt.tight_layout()plt.savefig('eda/plots/normal_data_histograms.png', dpi=150, bbox_inches='tight')plt.close()# Time series plotsfig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)axes[0].plot(normal['timestamp'], normal['temperature'], alpha=0.6, linewidth=0.5, color='blue')axes[0].set_title('Temperature Over Time (normal_data.csv)')axes[0].set_ylabel('Temperature (°C)')axes[0].grid(True, alpha=0.3)axes[1].plot(normal['timestamp'], normal['humidity'], alpha=0.6, linewidth=0.5, color='orange')axes[1].set_title('Humidity Over Time (normal_data.csv)')axes[1].set_xlabel('Timestamp')axes[1].set_ylabel('Humidity (% RH)')axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.savefig('eda/plots/normal_data_timeseries.png', dpi=150, bbox_inches='tight')plt.close()plt.figure(figsize=(8, 6))plt.scatter(normal['temperature'], normal['humidity'], alpha=0.3, s=5, c='purple')plt.title('Temperature vs Humidity (normal_data.csv)')plt.xlabel('Temperature (°C)')plt.ylabel('Humidity (% RH)')plt.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('eda/plots/normal_data_scatter.png', dpi=150, bbox_inches='tight')plt.close()plt.figure(figsize=(6, 5))corr = normal[['temperature', 'humidity']].corr()sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, square=True, fmt='.3f')plt.title('Correlation Heatmap (normal_data.csv)')plt.tight_layout()plt.savefig('eda/plots/normal_data_correlation.png', dpi=150, bbox_inches='tight')plt.close()plt.figure(figsize=(10, 5))intervals = normal['interval'].dropna()plt.hist(intervals[intervals <= 10], bins=20, alpha=0.7, edgecolor='black', color='teal')plt.axvline(2.0, color='red', linestyle='--', linewidth=2, label='Expected: 2.0s')plt.title('Sampling Interval Distribution (normal_data.csv, intervals ≤ 10s)')plt.xlabel('Interval (seconds)')plt.ylabel('Frequency')plt.legend()plt.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('eda/plots/normal_data_intervals.png', dpi=150, bbox_inches='tight')plt.close()# ============================================================# VISUALIZATIONS FOR pilot_data.csv# ============================================================fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)axes[0].plot(pilot['timestamp'], pilot['temperature'], alpha=0.7, linewidth=0.8, color='blue')axes[0].set_title('Temperature Over Time (pilot_data.csv)')axes[0].set_ylabel('Temperature (°C)')axes[0].grid(True, alpha=0.3)axes[1].plot(pilot['timestamp'], pilot['humidity'], alpha=0.7, linewidth=0.8, color='orange')axes[1].set_title('Humidity Over Time (pilot_data.csv)')axes[1].set_xlabel('Timestamp')axes[1].set_ylabel('Humidity (% RH)')axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.savefig('eda/plots/pilot_data_timeseries.png', dpi=150, bbox_inches='tight')plt.close()# Anomaly detail plots for pilot datafig, axes = plt.subplots(2, 2, figsize=(14, 10))# Event 1: 23:09:38 - 23:10:30event1 = pilot[(pilot['timestamp'] >= '2026-10-27 23:09:00') & (pilot['timestamp'] <= '2026-10-27 23:11:00')]axes[0, 0].plot(event1['timestamp'], event1['temperature'], 'o-', color='blue', label='Temperature')axes[0, 0].set_title('Event 1: Temperature (23:09 - 23:11)')axes[0, 0].set_ylabel('Temperature (°C)')axes[0, 0].grid(True, alpha=0.3)axes[0, 0].tick_params(axis='x', rotation=45)axes[0, 1].plot(event1['timestamp'], event1['humidity'], 'o-', color='orange', label='Humidity')axes[0, 1].set_title('Event 1: Humidity (23:09 - 23:11)')axes[0, 1].set_ylabel('Humidity (% RH)')axes[0, 1].grid(True, alpha=0.3)axes[0, 1].tick_params(axis='x', rotation=45)# Event 2: 00:34:20 - 00:34:56event2 = pilot[(pilot['timestamp'] >= '2026-10-28 00:33:30') & (pilot['timestamp'] <= '2026-10-28 00:35:30')]axes[1, 0].plot(event2['timestamp'], event2['temperature'], 'o-', color='blue')axes[1, 0].set_title('Event 2: Temperature (00:33:30 - 00:35:30)')axes[1, 0].set_ylabel('Temperature (°C)')axes[1, 0].grid(True, alpha=0.3)axes[1, 0].tick_params(axis='x', rotation=45)axes[1, 1].plot(event2['timestamp'], event2['humidity'], 'o-', color='orange')axes[1, 1].set_title('Event 2: Humidity (00:33:30 - 00:35:30)')axes[1, 1].set_xlabel('Timestamp')axes[1, 1].set_ylabel('Humidity (% RH)')axes[1, 1].grid(True, alpha=0.3)axes[1, 1].tick_params(axis='x', rotation=45)plt.tight_layout()plt.savefig('eda/plots/pilot_data_anomaly_detail.png', dpi=150, bbox_inches='tight')plt.close()# ============================================================# VISUALIZATIONS FOR test.csv# ============================================================fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)# Mark anomaly windowanomaly_start = pd.to_datetime('2026-10-27 22:03:29')anomaly_end = pd.to_datetime('2026-10-27 22:03:47')axes[0].plot(test['timestamp'], test['temperature'], 'o-', color='blue', markersize=4, label='Temperature')axes[0].axvspan(anomaly_start, anomaly_end, alpha=0.3, color='red', label='Anomaly Window')axes[0].set_title('Temperature Over Time (test.csv) - Injected Anomaly')axes[0].set_ylabel('Temperature (°C)')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[0].tick_params(axis='x', rotation=45)axes[1].plot(test['timestamp'], test['humidity'], 'o-', color='orange', markersize=4, label='Humidity')axes[1].axvspan(anomaly_start, anomaly_end, alpha=0.3, color='red', label='Anomaly Window')axes[1].set_title('Humidity Over Time (test.csv) - Injected Anomaly')axes[1].set_xlabel('Timestamp')axes[1].set_ylabel('Humidity (% RH)')axes[1].legend()axes[1].grid(True, alpha=0.3)axes[1].tick_params(axis='x', rotation=45)plt.tight_layout()plt.savefig('eda/plots/test_data_timeseries.png', dpi=150, bbox_inches='tight')plt.close()# Anomaly detailfig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)anomaly_data = test[(test['timestamp'] >= anomaly_start) & (test['timestamp'] <= anomaly_end)]axes[0].plot(anomaly_data['timestamp'], anomaly_data['temperature'], 'o-', color='red', markersize=6)axes[0].set_title('Injected Anomaly Detail: Temperature')axes[0].set_ylabel('Temperature (°C)')axes[0].grid(True, alpha=0.3)axes[0].tick_params(axis='x', rotation=45)axes[1].plot(anomaly_data['timestamp'], anomaly_data['humidity'], 'o-', color='red', markersize=6)axes[1].set_title('Injected Anomaly Detail: Humidity')axes[1].set_xlabel('Timestamp')axes[1].set_ylabel('Humidity (% RH)')axes[1].grid(True, alpha=0.3)axes[1].tick_params(axis='x', rotation=45)plt.tight_layout()plt.savefig('eda/plots/test_data_anomaly_detail.png', dpi=150, bbox_inches='tight')plt.close()print('All plots saved to eda/plots/')

Saved: eda/plots/normal_data_histograms.pngSaved: eda/plots/normal_data_boxplots.pngSaved: eda/plots/normal_data_timeseries.pngSaved: eda/plots/normal_data_scatter.pngSaved: eda/plots/normal_data_correlation.pngSaved: eda/plots/normal_data_intervals.pngSaved: eda/plots/pilot_data_timeseries.pngSaved: eda/plots/pilot_data_anomaly_detail.pngSaved: eda/plots/test_data_timeseries.pngSaved: eda/plots/test_data_anomaly_detail.png

## 9. Cross-Dataset ComparisonLet's compare the statistical properties across all three sensor datasets to understand if they represent the same operating conditions.

In [9]:
# Compare temperature statstemp_compare = pd.DataFrame({    'normal_data': normal['temperature'].describe(),    'pilot_data': pilot['temperature'].describe(),    'test_data': test['temperature'].describe()})print('=== TEMPERATURE COMPARISON ===')print(temp_compare)print()hum_compare = pd.DataFrame({    'normal_data': normal['humidity'].describe(),    'pilot_data': pilot['humidity'].describe(),    'test_data': test['humidity'].describe()})print('=== HUMIDITY COMPARISON ===')print(hum_compare)

=== CROSS-DATASET COMPARISON ===                    normal_data     pilot_data        test_datacount         49728.000000  3087.000000       56.000000mean            25.808518     27.291480      27.114286std              0.736721      1.470235       2.358692min             22.400000     26.000000      26.20000025%             25.800000     26.600000      26.20000050%             26.000000     26.800000      26.20000075%             26.200000     26.900000      26.200000max             27.000000     32.900000      34.200000                    normal_data     pilot_data        test_datacount         49728.000000  3087.000000       56.000000mean            62.072977     57.540460      62.128571std              4.258885      5.231717       6.579824min             46.800000     44.400000      59.30000025%             61.000000     57.600000      59.40000050%             63.500000     58.800000      59.50000075%             64.400000     60.000000      59.625000max             97.200000     

### Cross-Dataset Comparison Insights:**Temperature:**- normal_data: Mean 25.8°C, tight range (22.4-27.0°C)- pilot_data: Mean 27.3°C, wider range (26.0-32.9°C) due to anomalies- test_data: Mean 27.1°C, but 75th percentile is 26.2°C (anomaly inflates mean)- **Normal baseline appears to be ~26.0-26.2°C** across datasets**Humidity:**- normal_data: Mean 62.1%, range 46.8-97.2%- pilot_data: Mean 57.5%, range 44.4-95.8%- test_data: Mean 62.1%, but 75th percentile is 59.6% (anomaly inflates mean)- **Normal baseline appears to be ~59-60%** for pilot/test, but ~62-64% for normal_data**Key finding:** The datasets were collected under **different environmental conditions**. normal_data.csv shows higher humidity baseline. This means:- A model trained on normal_data.csv may not generalize to pilot/test conditions- We need to either: (a) collect more data matching deployment conditions, or (b) use domain adaptation- For now, normal_data.csv is the largest "normal" dataset but may have different operating conditions

## 10. Statistical Anomaly InvestigationLet's apply multiple statistical methods to identify potential outliers in normal_data.csv.

In [10]:
# IQR Outliersq1_t, q3_t = normal['temperature'].quantile([0.25, 0.75])iqr_t = q3_t - q1_tlower_t = q1_t - 1.5 * iqr_tupper_t = q3_t + 1.5 * iqr_tq1_h, q3_h = normal['humidity'].quantile([0.25, 0.75])iqr_h = q3_h - q1_hlower_h = q1_h - 1.5 * iqr_hupper_h = q3_h + 1.5 * iqr_htemp_iqr_low = normal[normal['temperature'] < lower_t]temp_iqr_high = normal[normal['temperature'] > upper_t]hum_iqr_low = normal[normal['humidity'] < lower_h]hum_iqr_high = normal[normal['humidity'] > upper_h]print('=== IQR OUTLIERS (normal_data.csv) ===')print(f'Temperature IQR outliers: {len(temp_iqr_low) + len(temp_iqr_high)}')print(f'  Below Q1-1.5*IQR ({lower_t:.1f}): {len(temp_iqr_low)}')print(f'  Above Q3+1.5*IQR ({upper_t:.1f}): {len(temp_iqr_high)}')print(f'Humidity IQR outliers: {len(hum_iqr_low) + len(hum_iqr_high)}')print(f'  Below Q1-1.5*IQR ({lower_h:.1f}): {len(hum_iqr_low)}')print(f'  Above Q3+1.5*IQR ({upper_h:.1f}): {len(hum_iqr_high)}')print()# Z-Score Outliers (|z| > 3)normal['temp_zscore'] = (normal['temperature'] - normal['temperature'].mean()) / normal['temperature'].std()normal['hum_zscore'] = (normal['humidity'] - normal['humidity'].mean()) / normal['humidity'].std()temp_z_out = normal[normal['temp_zscore'].abs() > 3]hum_z_out = normal[normal['hum_zscore'].abs() > 3]print('=== Z-SCORE OUTLIERS (|z| > 3) ===')print(f'Temperature Z-score outliers: {len(temp_z_out)}')print(f'Humidity Z-score outliers: {len(hum_z_out)}')print()print('Temperature Z-score outliers:')print(temp_z_out[['timestamp', 'temperature', 'humidity', 'temp_zscore']])print()print('Humidity Z-score outliers:')print(hum_z_out[['timestamp', 'temperature', 'humidity', 'hum_zscore']])

=== IQR OUTLIERS (normal_data.csv) ===Temperature IQR outliers: 6447  Below Q1-1.5*IQR (25.2): 6447  Above Q3+1.5*IQR (26.8): 0Humidity IQR outliers: 8132  Below Q1-1.5*IQR (55.9): 427  Above Q3+1.5*IQR (69.5): 7705=== Z-SCORE OUTLIERS (|z| > 3) ===Temperature Z-score outliers: 12Humidity Z-score outliers: 14Temperature Z-score outliers:                 timestamp  temperature  humidity  temp_zscore49717 2026-11-01 17:51:08         22.4      92.0    -4.63009349718 2026-11-01 17:51:10         22.4      91.8    -4.63009349719 2026-11-01 17:51:12         22.4      91.7    -4.63009349720 2026-11-01 17:51:14         22.4      91.5    -4.63009349721 2026-11-01 17:51:16         22.4      91.3    -4.63009349722 2026-11-01 17:51:18         22.4      91.1    -4.63009349723 2026-11-01 17:51:20         22.4      90.9    -4.63009349724 2026-11-01 17:51:22         22.4      90.7    -4.63009349725 2026-11-01 17:51:24         22.4      90.6    -4.63009349726 2026-11-01 17:51:26         22.4      90.4  

### Statistical Outlier Interpretation:**IQR Method (robust, doesn't assume Gaussian):**- Temperature: **6,447 low outliers** (below 25.2°C), 0 high outliers- Humidity: **427 low outliers** (below 55.9%), **7,705 high outliers** (above 69.5%)- The high humidity outliers (7,705 = 15.5% of data!) suggest the IQR upper fence (69.5%) is too conservative for this sensor/environment**Z-Score Method (assumes Gaussian, sensitive to outliers):**- Temperature: 12 outliers (all at the very end: 2026-11-01 17:51:08 - 17:51:32)- Humidity: 14 outliers (same time period + one at 2026-10-31 20:22:22)- **All Z-score outliers occur in the final minutes of the dataset** (last ~24 seconds)- These show **temperature dropping to 22.4°C** while **humidity spikes to 90-92%****Critical observation:** The last ~12 readings (24 seconds) show a simultaneous temperature crash AND humidity spike. This is physically unusual (cold air holds less moisture). This could be:1. Sensor failure / condensation on sensor2. Environmental event (AC turning on near humidifier?)3. Sensor artifact from power cycling**These 12-14 rows at the end of normal_data.csv are highly suspicious and should be investigated before using this data for training.**

## 11. Summary and Recommendations### Dataset Inventory Summary| Dataset | Rows | Header | Duplicates | Key Issues ||---------|------|--------|------------|------------|| normal_data.csv | 49,728 | ❌ Missing | 12,776 exact, 13,065 timestamp | Massive duplication, 39.5hr gap, suspicious tail| pilot_data.csv | 3,087 | ✅ Yes | 0 | Two clear anomaly events, 64min gap| test.csv | 56 | ✅ Yes | 0 | Clean, injected anomaly (10 rows)| test_labels.csv | 1 | ✅ Yes | N/A | Event-level label (not row-level)### Important Statistical Findings1. **normal_data.csv normal baseline**: Temp ~25.8°C (IQR 25.8-26.2), Humidity ~62.1% (IQR 61.0-64.4)2. **Massive duplication**: 26.3% of rows have duplicate timestamps; up to 80 rows per timestamp3. **Large logging gaps**: Up to 39.5 hours; data is not continuous4. **Suspicious tail**: Last 12 rows show temp crash (22.4°C) + humidity spike (90-92%)5. **Cross-dataset shift**: normal_data humidity baseline (~62%) differs from pilot/test (~59-60%)6. **Pilot anomalies**: Two real events with simultaneous temp+humidity spikes7. **Test anomaly**: Sharp injected spike (temp +8°C, hum +23%) over 18 seconds### Duplicate Analysis Summary- **Exact duplicate rows**: 12,761 (25.7% of data)- **Duplicate timestamps**: 13,065 (26.3% of data)- **Top duplicate timestamp**: 80 rows at 2026-10-30 18:33:40- **Pattern**: Clusters of 50-54 rows at same timestamp suggest logging bug- **Different values at same timestamp**: 304 rows (13,065 - 12,761)### Timestamp and Logging Problems1. **Zero-second intervals**: 13,065 occurrences (duplicate timestamps)2. **Expected 2s intervals**: 35,727 (71.8%)3. **3-second intervals**: 914 (1.8%)4. **Large gaps**: 10 gaps > 10 minutes, max 39.5 hours5. **Missing header**: normal_data.csv lacks header row### Potential Anomaly Periods1. **Pilot Event 1**: 2026-10-27 23:09:38-23:10:30 (humidity spike to 95%)2. **Pilot Event 2**: 2026-10-28 00:34:20-00:34:56 (temp to 32.9°C, hum to 95.8%)3. **Test Injected**: 2026-10-27 22:03:29-22:03:47 (temp to 34.2°C, hum to 82.1%)4. **Normal data tail**: 2026-11-01 17:51:08-17:51:32 (temp 22.4°C, hum 90-92%)### Test-Label Interpretation- **Event-level label**: Single row in test_labels.csv marks 18-second window- **10 test rows** fall in this window (indices 37-46)- **46 test rows** are normal baseline- **Must expand to row-level** for evaluation metrics### Dataset Suitability for ML Stages| Stage | Recommended Dataset | Notes ||-------|---------------------|-------|| Training (normal) | normal_data.csv (cleaned) | Remove duplicates, investigate tail, verify conditions match deployment || Validation | pilot_data.csv | Contains real anomalies - good for threshold tuning || Testing | test.csv + expanded labels | Small but clean; only for final evaluation |### Recommended Data Cleaning Decisions1. **DO NOT MODIFY ORIGINAL CSVs** - keep raw data immutable2. **Create cleaned versions** in `data/processed/`:   - Remove exact duplicate rows (keep first)   - For duplicate timestamps with different values: investigate, possibly keep first or average   - **Flag but don't auto-remove** the suspicious tail (last 12 rows)   - Add sequence numbers to detect dropped packets3. **Handle large gaps**: Don't interpolate across >1 hour gaps4. **Align datasets**: Consider if normal_data.csv conditions match deployment5. **Expand test labels**: Create row-level labels from event-level for evaluation---